In [ ]:
    #CreateDataset.ipynb

In [11]:
import os
import cv2
from tqdm import tqdm
import json

In [12]:
def load_json_file(file_path):
    with open(file_path, 'r') as file:
        data = json.load(file)
    return data
lis=load_json_file("metadata.json")

In [13]:
lis

{'zumqqvixhu.mp4': {'label': 'FAKE',
  'split': 'train',
  'original': 'hntguogkqd.mp4'},
 'utdlsqfykm.mp4': {'label': 'FAKE',
  'split': 'train',
  'original': 'nswtvttxre.mp4'},
 'mdfndlljvt.mp4': {'label': 'FAKE',
  'split': 'train',
  'original': 'ptkcmwnfjv.mp4'},
 'maktypgsfl.mp4': {'label': 'FAKE',
  'split': 'train',
  'original': 'objgwnmscm.mp4'},
 'pleqihjpif.mp4': {'label': 'FAKE',
  'split': 'train',
  'original': 'xrhqtmxlvx.mp4'},
 'yejvlyggtw.mp4': {'label': 'FAKE',
  'split': 'train',
  'original': 'mwwploizlj.mp4'},
 'yotsfuryir.mp4': {'label': 'FAKE',
  'split': 'train',
  'original': 'dvwpvqdflx.mp4'},
 'tguqyatciq.mp4': {'label': 'FAKE',
  'split': 'train',
  'original': 'lujvyveojc.mp4'},
 'fjzrvkleur.mp4': {'label': 'FAKE',
  'split': 'train',
  'original': 'gylcfcozce.mp4'},
 'kylqyoxeqm.mp4': {'label': 'FAKE',
  'split': 'train',
  'original': 'zwswwwrefl.mp4'},
 'txvcwipjsf.mp4': {'label': 'FAKE',
  'split': 'train',
  'original': 'rmpbqeawlq.mp4'},
 'ehthgupu

In [14]:
name_list=[]
label_list=[]
for i in lis:
    name_list.append(i)
    if lis[i]['label']=="FAKE":
        label_list.append(1)
        name_list.append(lis[i]['original'])
        label_list.append(0)
    else:
        label_list.append(0)

In [5]:
def process_videos(video_folder, output_folder,name_list,label_list):
    
    output_fake=os.path.join(output_folder,"fake")
    output_org=os.path.join(output_folder,"real")
    
    if not os.path.exists(output_fake):
        os.makedirs(output_fake)
    if not os.path.exists(output_org):
        os.makedirs(output_org)

    face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades+'haarcascade_frontalface_default.xml')

    video_files = [f for f in os.listdir(video_folder) if f.endswith(('.mp4', '.avi', '.mov'))]
    j=0
    for video_file in tqdm(video_files):
        if video_file in name_list:
            if label_list[name_list.index(video_file)]:
                output_path=output_fake
            else:
                output_path=output_org
            video_path = os.path.join(video_folder, video_file)
            cap = cv2.VideoCapture(video_path)
            total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
            frame_count = 0
            while True:
                ret, frame = cap.read()
                if not ret:
                    break

                gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

                faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(30, 30))

                for i, (x, y, w, h) in enumerate(faces):
                    face = frame[y:y+h, x:x+w]
                    face_name = "video"+str(j)+"frame_"+str(frame_count)+"_face_"+str(i)+".png"
                    face_path = os.path.join(output_path, face_name)
                    cv2.imwrite(face_path, face)

                frame_count += 1

            cap.release()
            j=j+1

In [15]:
import os
import cv2
import face_recognition
from tqdm import tqdm

def process_videos(video_folder, output_folder, name_list, label_list):
    
    output_fake = os.path.join(output_folder, "fake")
    output_org  = os.path.join(output_folder, "real")
    
    os.makedirs(output_fake, exist_ok=True)
    os.makedirs(output_org, exist_ok=True)

    video_files = [f for f in os.listdir(video_folder) 
                   if f.endswith(('.mp4', '.avi', '.mov'))]

    j = 0
    for video_file in tqdm(video_files):
        if video_file in name_list:
            
            if label_list[name_list.index(video_file)]:
                output_path = output_fake
            else:
                output_path = output_org

            video_path = os.path.join(video_folder, video_file)
            cap = cv2.VideoCapture(video_path)

            frame_count = 0
            while True:
                ret, frame = cap.read()
                if not ret:
                    break

                # Convert BGR → RGB
                rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

                # Detect faces
                face_locations = face_recognition.face_locations(
                    rgb_frame, model="hog"   # use "cnn" for higher accuracy (GPU)
                )

                for i, (top, right, bottom, left) in enumerate(face_locations):
                    face = frame[top:bottom, left:right]

                    face_name = f"video{j}_frame_{frame_count}_face_{i}.png"
                    face_path = os.path.join(output_path, face_name)

                    cv2.imwrite(face_path, face)

                frame_count += 1

            cap.release()
            j += 1

In [ ]:
video_folder = "/Users/ardrapadmakumar/Desktop/deepfake/deepfake/dfdc_train_part_1"
output_folder = "/Users/ardrapadmakumar/Desktop/deepfake/deepfake/op"
process_videos(video_folder, output_folder,name_list,label_list)

In [ ]:
!pip install opencv-python

In [ ]:
!pip install face_recognition

In [ ]:
!pip install dlib